<a href="https://colab.research.google.com/github/AbdAlRahman-Odeh-99/Two_Phases_Simulation/blob/main/constrained_supervised_training_submodular_gmm2c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Multi-View Resource Allocation Simulation

This notebook simulates a multi-view resource allocation problem, where an agent learns to select a subset of 'views' (features) to minimize prediction error under a budget constraint. It compares a dynamic learning algorithm against an optimal static solution.

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import scipy.optimize as opt
from scipy.stats import norm
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

### Global Parameters

These parameters control the simulation and can be adjusted.

In [12]:
# Simulation parameters (converted from argparse arguments)
seed = 42 # random seed for simulation
num_views = 40 # number of views (features) - restored to original script's default
num_rounds = 1000 # number of simulations
num_trials = 20 # number of trials
num_samples = 1000 # number of samples
budget_ratio = 0.7 # budget ratio
output_file_name = "output_gmm2_su.csv" # output file name
no_opt = True # skip computing the optimal solution (restored to original script's default)

### Helper Functions

These functions define the core logic for data generation, optimal solution calculation, subset selection, and label matching.

In [13]:
def generate_data(rng,nsamples=1000,nclasses=2,nviews=10,seed=42):
    mm = rng.random(size=(nclasses,nviews)) * 2 # symmetric
    # shared variances
    # FIXME: currently simplify to unit variances, only means differ
    stds = 1.0 # simplification
    #mm = np.concatenate([-mm, mm],axis=0) #
    data = make_blobs(nsamples,n_features=nviews,centers=mm,cluster_std=stds,random_state=seed)
    return data[0], mm, data[1]

In [14]:
def optimal_static(means,stds,costs,budget,num_rounds):
    # given the means (symmetric) and costs
    # compute the error probability and let the accuracy (1-err) be the expected reward
    # it is the qfunction of mean squares divided by the standard variances
    nviews = len(costs)
    # the statistics are
    # mean differences
    mean_diff_sq = np.square( 0.5*(means[0,:] - means[1,:])/stds )
    #stats = np.square(means/stds)
    lp_table = np.zeros((2**nviews-1,2)) # (expected reward, costs)
    for comb in range(1,2**nviews):
        sel_idx = np.array([int(bit) for bit in f"{comb:0{nviews}b}"]).astype(bool)
        snrs = np.sum(mean_diff_sq[sel_idx])
        err_rate_calc = norm.sf(np.sqrt(snrs), loc=0,scale=1) # survival function

        cost_sum = np.sum(costs[sel_idx])
        # offset the index by one as range starts from 1 instead of 0
        lp_table[comb-1,:] = np.array([1-err_rate_calc,cost_sum])

    # use linear program to solve the optimal constrained training
    budget_ratio = budget / num_rounds
    r_expect = np.expand_dims(lp_table[:,0],axis=0)
    r_cost = np.expand_dims(lp_table[:,1],axis=0)
    simplex_sum = np.ones((1,2**nviews-1))
    x_bound = [(0,None) for _ in range(2**nviews-1)]
    res = opt.linprog(-r_expect,A_ub=r_cost,b_ub=budget_ratio,A_eq=simplex_sum, b_eq=1,bounds=x_bound)
    rt_msg = res.fun
    opt_pi = res.x # this is the optimal static distribution
    # provide the planning output as the selection distribution for the subsets
    opt_reward = num_rounds * r_expect @ opt_pi
    opt_costs = num_rounds * r_cost @ opt_pi

    print(f"Rounds:{num_rounds}, Budget:{budget}, Optimal Reward: {opt_reward.item():.4e}, Optimal_cost: {opt_costs.item():.4e}")
    return {"solver_message":rt_msg,"reward":opt_reward.item(),"cost":opt_costs.item(),"solution":opt_pi}

In [15]:
def get_subset(est_means_sq,costs,lambda_dual,spending_ratio):
    best_reward = -np.inf
    best_subset = None
    nviews = len(costs)
    #est_means_sq = np.square(0.5 * (est_means[0,:] - est_means[1,:])/1.0) # assume std =1.0
    for v in range(1, 2**nviews):
        subset = np.array([int(bit) for bit in f"{v:0{nviews}b}"]).astype(bool)
        tmp_cost = np.sum(costs[subset])
        snrs = est_means_sq[subset]
        tmp_err_rate = norm.sf(np.sqrt(np.sum(snrs)), loc=0, scale=1)
        tmp_reward = 1-tmp_err_rate - lambda_dual * (tmp_cost - spending_ratio)
        if tmp_reward > best_reward:
            best_reward = tmp_reward
            best_subset = subset
    # greedily choose the maximum
    return best_subset

In [ ]:
import numba
import numpy as np
from scipy.special import erf # Import erf directly from scipy.special

# Moved gain_func back to module level and re-jitted
@numba.njit
def gain_func(snrs):
  # norm.sf(x) = 0.5 * erfc(x / sqrt(2))
  # erfc(x) = 1 - erf(x)
  # So, norm.sf(x) = 0.5 * (1 - erf(x / sqrt(2)))
  # Here x = 0.5 * np.sqrt(np.sum(snrs))
  arg = (0.5 * np.sqrt(np.sum(snrs))) / np.sqrt(2.0)
  error_rate = 0.5 * (1 - erf(arg)) # Call erf directly
  return 1 - error_rate

# greedy approximate oracle
def greedy_oracle(est_means_sq,costs,lambda_dual,remaining_budget):
    num_elements = len(costs)
    sel_set = []
    current_cost = 0.0
    current_objective = 0.0
    available_elements = set(range(num_elements))

    # harvest free views first (iterate over a copy)
    for i in list(available_elements):
        cost_i = costs[i]
        if cost_i == 0:
            tmp_select = sel_set + [i]
            margin_gain = gain_func(est_means_sq[np.array(tmp_select)]) - current_objective

            if margin_gain > 0:
                sel_set.append(i)
                current_objective += margin_gain
                available_elements.remove(i)

    free_indices = list(sel_set)

    while available_elements and current_cost < remaining_budget:
        best_margin = -1
        best_add = None

        for i in available_elements:
            cost_i = costs[i]
            if current_cost + cost_i <= remaining_budget:
                tmp_select = sel_set + [i]
                margin_gain = gain_func(est_means_sq[np.array(tmp_select)]) - current_objective
                # now there is no zero cost elements, add epsilon for safety
                gain_ratio = margin_gain / (cost_i + 1e-9)
                # only add it if the margin is greater than the shadow price
                if gain_ratio > lambda_dual and gain_ratio > best_margin:
                    best_margin = gain_ratio
                    best_add = i
        if best_add is not None:
            sel_set.append(best_add)
            current_cost += costs[best_add]
            current_objective = gain_func(est_means_sq[np.array(sel_set)])
            available_elements.remove(best_add)
        else:
            break
    # Giant item check
    greedy_solution_reward = current_objective
    greedy_solution_indices = list(sel_set)

    best_single_item = None
    best_giant_reward = -np.inf
    for i in range(num_elements):
        if i in free_indices:
            continue
        cost_i = costs[i]
        if 0 < cost_i <= remaining_budget:
            tmp_giant_indices = free_indices + [i]
            reward_with_giant = gain_func(est_means_sq[np.array(tmp_giant_indices)])
            if reward_with_giant > best_giant_reward:
                best_giant_reward = reward_with_giant
                best_single_item = i

    # compare against the giant element
    if best_single_item is not None and best_giant_reward > greedy_solution_reward:
        final_indices = free_indices + [best_single_item]
    else:
        final_indices = greedy_solution_indices
    # convert to boolean vector
    return np.array([idx in final_indices for idx in range(num_elements)]).astype("bool")

In [17]:
def label_matching(y_true, y_pred):
    assert(len(y_pred) == len(y_true))
    # 1. Compute confusion matrix
    D = confusion_matrix(y_true, y_pred)
    # 2. Solve the linear sum assignment problem
    # We negate the matrix because the algorithm minimizes cost, and we want to maximize matches
    row_ind, col_ind = linear_sum_assignment(D.max() - D)
    # 3. Create a mapping from predicted to true labels
    mapping = {col: row for row, col in zip(row_ind, col_ind)}
    # 4. Apply the mapping to the predictions
    y_pred_mapped = np.array([mapping[label] for label in y_pred])
    return y_pred_mapped, mapping

In [18]:
def simulation(num_views,costs,num_rounds,budget,x_data,y_labels):
    # initialize the estimators
    # for simplicity, assume estimating means only
    est_means = np.array(np.concatenate([np.zeros((1,num_views)),-np.ones((1,num_views))],axis=0))
    est_counts = np.ones((2,num_views)) # add a dummy count

    remain_budget = budget
    spending_ratio = budget / num_rounds
    acc_reward = 0
    omd_lambda = 0
    lambda_max = 10 # change this when needed
    step_size = 1.0
    noise_var = 1.0
    # optimistic
    xi_fixed = 2.0

    # recording
    record_acc = np.zeros((num_rounds))
    record_pred = np.zeros((num_rounds))
    for nr in range(num_rounds):
        # add the optimistic means
        mean_diff = est_means[0,:] - est_means[1,:]
        optimistic_means_sq = 0.25 * np.square(mean_diff/noise_var) + np.sqrt(xi_fixed * np.log(nr+1)/est_counts.sum(axis=0))
        #subset = get_subset(optimistic_means_sq,costs,omd_lambda,spending_ratio)
        subset = greedy_oracle(optimistic_means_sq,costs,omd_lambda,remain_budget)
        if remain_budget <=0:
            #subset = [0] # only the free view is possible
            subset = np.zeros((num_views)).astype("bool")
        y_true = y_labels[nr] # NOTE: only for evaluation, not involved in training
        x_observe = x_data[nr,subset]

        # prediction
        # Use all features for prediction, with non-observed ones being 0
        boundary = 0.5 * (est_means[0,subset] + est_means[1,subset])
        mean_diff = est_means[0,subset] - est_means[1,subset]
        y_pred = 0 if np.sum( (boundary - x_observe) * mean_diff) >= 0 else 1
        record_pred[nr] = y_pred
        #acc_reward += int(y_pred == y_true)

        # algorithm update
        instant_cost = np.sum(costs[subset])
        remain_budget = max(0, remain_budget - instant_cost)
        # parameter update
        scaler = np.squeeze(1 - 1/(est_counts[y_true,subset]+1))
        est_means[y_true,subset] = scaler * est_means[y_true,subset] + (1-scaler) * x_observe
        est_counts[y_true,subset] += 1


        raw_lambda = omd_lambda + step_size * (instant_cost - spending_ratio)
        omd_lambda = max(0, min(lambda_max, raw_lambda))
        # recording
        #record_acc[nr] = acc_reward/ (nr+1)

    # do a label matching here
    y_pred_mapped, _ = label_matching(y_labels,record_pred)
    acc_reward = np.sum(y_pred_mapped == y_labels)
    record_acc = np.cumsum(y_pred_mapped == y_labels) / np.arange(1,num_rounds+1)


    avg_acc = acc_reward / num_rounds
    return {"reward": acc_reward, "avg_acc":avg_acc, "record_acc":record_acc, "spending":budget - remain_budget}

### Main Simulation Loop

This section runs the simulation for a specified number of trials and collects the results.

In [19]:
rng = np.random.default_rng(seed=seed)
result_pd = {"trial":[], "reward":[], "spending":[], "avg_acc":[],"optimal_reward":[]}

if num_views > 10 and not no_opt:
  print(f"[WARN] For num_views={num_views}, optimal solution requires slow power set enumeration. Set `no_opt = True` to skip.")

for trial in range(num_trials):
    print(f"Running Trial {trial + 1}/{num_trials}...") # Commented out to reduce I/O overhead
    budget = num_rounds * budget_ratio
    costs = rng.random(size=(num_views,))
    # create free view
    costs[0] = 0
    # normalize views to sum to 1
    costs = costs/np.sum(costs)
    t_data, t_means, t_labels = generate_data(rng,num_samples,2,num_views,seed)
    if no_opt:
        opt_dict = {"reward":0} # dummy reward
    else:
        opt_dict = optimal_static(t_means,1.0,costs,budget,num_rounds)

    sim_dict = simulation(num_views,costs,num_rounds,budget,t_data,t_labels)

    # accumulating
    result_pd['trial'].append(trial)
    result_pd['reward'].append(sim_dict['reward'])
    result_pd['spending'].append(sim_dict['spending'])
    result_pd['avg_acc'].append(sim_dict['avg_acc'])
    result_pd['optimal_reward'].append(opt_dict['reward'])

res_df = pd.DataFrame.from_dict(result_pd)

print("Simulation Results:")
display(res_df)

Simulation Results:


,trial,reward,spending,avg_acc,optimal_reward
0,0,971,591.266178,0.971,0
1,1,992,570.176955,0.992,0
2,2,985,586.638947,0.985,0
3,3,986,604.077148,0.986,0
4,4,990,593.212486,0.990,0
5,5,974,600.812975,0.974,0
6,6,985,573.863092,0.985,0
7,7,983,588.842165,0.983,0
8,8,968,596.018076,0.968,0
9,9,981,597.201919,0.981,0


### Summary of Results

The table above shows the simulation results for each trial, including the achieved reward, spending, average accuracy, and the optimal reward if calculated.